# Stage 3: energies and charge transfer integrals

For ten randomly chosen configurations of each motif, this computes the
electronic coupling between the frontier orbitals of the two monomers, together
with the site energies that go with it.

The coupling comes from dimer projection (DIPRO): the converged monomer
orbitals are projected onto the dimer Fock matrix,

$$J_{ab} = \langle \phi_a | \hat{F}_D | \phi_b \rangle, \qquad
  S_{ab} = \langle \phi_a | \phi_b \rangle, \qquad
  e_{a} = \langle \phi_a | \hat{F}_D | \phi_a \rangle$$

and the non-orthogonality of the two fragment orbitals is divided out:

$$J^{\rm eff} = \frac{J_{ab} - S_{ab}(e_a + e_b)/2}{1 - S_{ab}^2}$$

$J^{\rm eff}$ is the quantity to use downstream - the raw $J_{ab}$ is basis-set
dependent through that non-orthogonality, and at 3 A contact the correction
typically changes it by tens of percent.

**Two numbers per frame matter for transport, not one.** $J^{\rm eff}$ is the
coupling, and $\Delta e = e_a - e_b$ is the site energy difference across the
pair. A Marcus hopping rate needs both: $J$ sets the prefactor, $\Delta e$
enters the activation energy. Both fluctuate with the thermal disorder that
stage 2 sampled, so both are recorded per frame.

**LUMO-LUMO is the coupling that matters here.** AQx-2 is a non-fullerene
acceptor, so the acceptor phase carries electrons. HOMO-HOMO comes out of the
same SCFs at no extra cost and is recorded alongside it.

## Input

`configs_H_backbone/` and `configs_J_end_group/`, from stage 2.

## Output

Per motif: `transfer_integrals_<motif>.csv` with every quantity below, and
`J_<motif>.txt` with just the LUMO-LUMO couplings in meV, one per frame.

Needs `pyscf`, which is Linux/macOS only - under Windows run this from WSL:

```
pip install pyscf
```

In [ ]:
# ---- Settings -----------------------------------------------------------
import os
import time

import numpy as np
from pyscf import dft, gto, lib

MOTIFS = ["H_backbone", "J_end_group"]

N_SAMPLE = 10               # frames per motif, drawn from the 40 available
SEED = 0                    # fixed, so a rerun picks the same frames

BASIS = "def2-svp"          # no diffuse functions: at 3 A contact they cause
                            # linear dependence in the dimer basis for no gain
XC = "b3lyp"                # couplings are fairly insensitive to the
                            # functional, much less so to the basis; b3lyp is
                            # what most of the organic semiconductor transfer
                            # integral literature uses. Note pyscf's "b3lyp" is
                            # the VWN5 form, "b3lypg" reproduces Gaussian's

# "superposition" builds the dimer Fock matrix from the two converged monomer
# densities placed block-diagonally - the usual FODFT approximation, and a mild
# one for neutral closed-shell fragments. It skips the dimer SCF entirely.
# "dimer_scf" is the fully self-consistent result, and roughly three times the
# work.
MODE = "superposition"

CONV_TOL = 1e-7             # tighter than a Fock matrix element between
                            # frontier orbitals needs
GRID_LEVEL = 1              # default is 3; the coupling is insensitive to the
                            # XC quadrature at the meV level

N_THREADS = os.cpu_count()

HARTREE_TO_MEV = 27211.386245988
HARTREE_TO_EV = 27.211386245988

COLUMNS = ["config",
           "E_A_eV", "E_B_eV",
           "J_lumo_meV", "S_lumo", "e_a_lumo_eV", "e_b_lumo_eV",
           "dE_lumo_eV", "J_raw_lumo_meV",
           "J_homo_meV", "S_homo", "e_a_homo_eV", "e_b_homo_eV",
           "dE_homo_eV", "J_raw_homo_meV",
           "homo_A_eV", "lumo_A_eV", "homo_B_eV", "lumo_B_eV",
           "lumo_gap_A_eV", "lumo_gap_B_eV", "homo_gap_A_eV", "homo_gap_B_eV",
           "seconds"]
HEADER = ",".join(COLUMNS) + "\n"

lib.num_threads(N_THREADS)
print(f"basis {BASIS}, functional {XC}, mode {MODE}, {N_THREADS} threads")
print(f"motifs: {', '.join(MOTIFS)}, {N_SAMPLE} frames each")

In [ ]:
# ---- Helpers ------------------------------------------------------------
def load_xyz(path):
    """Symbols and coordinates from an xyz file.

    Parsed by hand rather than through a library because the comment line
    stage 2 writes is full of '=' and ',' characters that some readers try to
    interpret as extended-xyz key/value pairs.
    """
    with open(path) as f:
        n = int(f.readline().split()[0])
        f.readline()
        symbols, coords = [], []
        for _ in range(n):
            parts = f.readline().split()
            symbols.append(parts[0])
            coords.append([float(x) for x in parts[1:4]])
    return symbols, np.array(coords)


def build_mol(symbols, coords):
    mol = gto.Mole()
    mol.atom = [(s, tuple(c)) for s, c in zip(symbols, coords)]
    mol.basis = BASIS
    mol.charge, mol.spin = 0, 0
    mol.unit = "Angstrom"
    mol.verbose = 0
    mol.build()
    return mol


def make_mf(mol):
    """Density-fitted RKS. Density fitting is a large speedup here and pyscf
    keeps the three-index tensor on disk, so its size is not a constraint."""
    mf = dft.RKS(mol).density_fit()
    mf.xc = XC
    mf.conv_tol = CONV_TOL
    mf.grids.level = GRID_LEVEL
    mf.max_cycle = 100
    return mf


def run_scf(mol, dm0=None):
    """Converge an SCF, falling back to the second-order solver if the default
    DIIS iteration stalls."""
    mf = make_mf(mol)
    mf.kernel(dm0=dm0)
    if not mf.converged:
        mf = mf.newton()
        mf.kernel(mf.make_rdm1())
    if not mf.converged:
        raise RuntimeError(f"SCF did not converge for {mol.natm} atoms")
    return mf


def frontier(mf):
    """HOMO and LUMO indices, their orbital energies in eV, and the gap to the
    next orbital on each side.

    The gaps tell you whether a single-orbital coupling is adequate: acceptor
    LUMOs are often near-degenerate, and if LUMO and LUMO+1 sit within ~0.1 eV
    the 2x2 frontier block should be diagonalised instead.
    """
    homo = int(np.count_nonzero(mf.mo_occ > 0)) - 1
    lumo = homo + 1
    e = mf.mo_energy
    return (homo, lumo,
            e[homo] * HARTREE_TO_EV, e[lumo] * HARTREE_TO_EV,
            (e[homo] - e[homo - 1]) * HARTREE_TO_EV,
            (e[lumo + 1] - e[lumo]) * HARTREE_TO_EV)


def coupling(ca, cb, F, S):
    """Effective coupling between two fragment orbitals already expressed in
    the dimer AO basis. Returns (J_eff, S_ab, e_a, e_b, J_raw), in Hartree
    except S_ab."""
    j_raw = float(ca @ F @ cb)
    s_ab = float(ca @ S @ cb)
    e_a = float(ca @ F @ ca)
    e_b = float(cb @ F @ cb)
    j_eff = (j_raw - s_ab * (e_a + e_b) / 2) / (1 - s_ab ** 2)
    return j_eff, s_ab, e_a, e_b, j_raw


def process(config_dir, tag):
    """Two monomer SCFs, the dimer Fock matrix, then the projection.

    The fragment orbitals embed into the dimer AO basis by zero-padding: pyscf
    orders AOs by atom and the dimer file lists all of A before all of B, so
    the dimer AO space is exactly the direct sum of the monomer spaces in that
    order. Both facts are checked rather than trusted.
    """
    t0 = time.time()
    sym_d, pos_d = load_xyz(f"{config_dir}/{tag}_dimer.xyz")
    sym_a, pos_a = load_xyz(f"{config_dir}/{tag}_A.xyz")
    sym_b, pos_b = load_xyz(f"{config_dir}/{tag}_B.xyz")

    if sym_d != sym_a + sym_b:
        raise SystemExit(f"{tag}: dimer atom order is not A then B")
    if not np.allclose(pos_d, np.vstack([pos_a, pos_b]), atol=1e-6):
        raise SystemExit(f"{tag}: monomer coordinates differ from the dimer")

    mol_d = build_mol(sym_d, pos_d)
    mol_a = build_mol(sym_a, pos_a)
    mol_b = build_mol(sym_b, pos_b)
    na, nb = mol_a.nao, mol_b.nao
    if na + nb != mol_d.nao:
        raise SystemExit(f"{tag}: dimer basis is not the direct sum of the "
                         f"monomer bases ({na} + {nb} != {mol_d.nao})")

    mf_a, mf_b = run_scf(mol_a), run_scf(mol_b)

    # the superposed monomer density is both the starting guess for the dimer
    # SCF and, in superposition mode, the density the Fock matrix is built
    # from, so it is assembled either way
    dm_super = np.zeros((mol_d.nao, mol_d.nao))
    dm_super[:na, :na] = mf_a.make_rdm1()
    dm_super[na:, na:] = mf_b.make_rdm1()

    if MODE == "dimer_scf":
        F = run_scf(mol_d, dm0=dm_super).get_fock()
    else:
        F = make_mf(mol_d).get_fock(dm=dm_super)
    S = mol_d.intor("int1e_ovlp")

    homo_a, lumo_a, eh_a, el_a, hgap_a, lgap_a = frontier(mf_a)
    homo_b, lumo_b, eh_b, el_b, hgap_b, lgap_b = frontier(mf_b)

    out = {}
    for name, ia, ib in (("lumo", lumo_a, lumo_b), ("homo", homo_a, homo_b)):
        ca, cb = np.zeros(mol_d.nao), np.zeros(mol_d.nao)
        ca[:na] = mf_a.mo_coeff[:, ia]
        cb[na:] = mf_b.mo_coeff[:, ib]
        j_eff, s_ab, e_a, e_b, j_raw = coupling(ca, cb, F, S)
        out[name] = dict(J=j_eff * HARTREE_TO_MEV, S=s_ab,
                         e_a=e_a * HARTREE_TO_EV, e_b=e_b * HARTREE_TO_EV,
                         dE=(e_a - e_b) * HARTREE_TO_EV,
                         J_raw=j_raw * HARTREE_TO_MEV)

    dt = time.time() - t0
    values = [tag,
              f"{mf_a.e_tot * HARTREE_TO_EV:.6f}",
              f"{mf_b.e_tot * HARTREE_TO_EV:.6f}"]
    for name in ("lumo", "homo"):
        o = out[name]
        values += [f"{o['J']:.4f}", f"{o['S']:.6e}", f"{o['e_a']:.6f}",
                   f"{o['e_b']:.6f}", f"{o['dE']:.6f}", f"{o['J_raw']:.4f}"]
    values += [f"{eh_a:.6f}", f"{el_a:.6f}", f"{eh_b:.6f}", f"{el_b:.6f}",
               f"{lgap_a:.4f}", f"{lgap_b:.4f}", f"{hgap_a:.4f}",
               f"{hgap_b:.4f}", f"{dt:.1f}"]
    if len(values) != len(COLUMNS):
        raise SystemExit(f"row has {len(values)} fields, header has "
                         f"{len(COLUMNS)}")

    print(f"  {tag}  J(LUMO) {out['lumo']['J']:8.2f} meV  "
          f"dE(LUMO) {out['lumo']['dE']:+7.3f} eV  "
          f"J(HOMO) {out['homo']['J']:8.2f} meV  {dt / 60:.1f} min",
          flush=True)
    return ",".join(values) + "\n"

## Choose the frames

Ten of each motif's forty, drawn without replacement with a fixed seed so a
rerun reproduces the same selection. Random rather than the first ten because
consecutive frames early in a trajectory share whatever the equilibration
happened to leave behind, and the point is to sample the thermal ensemble.

In [ ]:
# ---- Sample frames ------------------------------------------------------
selection = {}
for motif in MOTIFS:
    config_dir = f"configs_{motif}"
    if not os.path.isdir(config_dir):
        raise SystemExit(f"{config_dir} not found - run stage 2 first")
    all_tags = sorted(f[:-10] for f in os.listdir(config_dir)
                      if f.endswith("_dimer.xyz"))
    rng = np.random.default_rng(SEED)
    selection[motif] = sorted(
        rng.choice(all_tags, size=min(N_SAMPLE, len(all_tags)),
                   replace=False).tolist())
    print(f"{motif}: {len(selection[motif])} of {len(all_tags)} frames")
    print("  " + ", ".join(selection[motif]))

## Run

Results are appended per frame and frames already in a motif's CSV are skipped,
so an interrupted run loses at most the frame in progress - rerun this cell and
it picks up where it stopped. Delete the CSVs to start over, and do delete them
if you change `SEED`, `N_SAMPLE` or any of the method settings.

In [ ]:
# ---- Couplings, one frame at a time -------------------------------------
for motif in MOTIFS:
    config_dir = f"configs_{motif}"
    output = f"transfer_integrals_{motif}.csv"
    print(f"\n{'=' * 70}\n{motif}\n{'=' * 70}", flush=True)

    done = set()
    if os.path.exists(output):
        with open(output) as f:
            done = {ln.split(",")[0] for ln in f.readlines()[1:] if ln.strip()}
        print(f"{len(done)} frames already done, skipping those", flush=True)
    else:
        with open(output, "w") as f:
            f.write(HEADER)

    for tag in selection[motif]:
        if tag in done:
            continue
        with open(output, "a") as f:
            f.write(process(config_dir, tag))

print("\ndone")

In [ ]:
# ---- Summary ------------------------------------------------------------
summary = {}
for motif in MOTIFS:
    rows = np.atleast_1d(np.genfromtxt(f"transfer_integrals_{motif}.csv",
                                       delimiter=",", names=True))
    summary[motif] = rows
    print(f"\n{motif}  ({len(rows)} frames)")
    for name in ("lumo", "homo"):
        j = np.abs(rows[f"J_{name}_meV"])
        # <J^2>^0.5 is the transport-relevant average: Marcus rates go as J^2,
        # so averaging J itself over a disordered ensemble understates it
        print(f"  |J| {name.upper():4s} mean {j.mean():6.1f}  sd {j.std():5.1f}"
              f"  rms {np.sqrt((j ** 2).mean()):6.1f} meV  "
              f"(min {j.min():.1f}, max {j.max():.1f})")
    de = rows["dE_lumo_eV"]
    print(f"  site energy difference, LUMO: mean {de.mean():+.3f} eV, "
          f"sd {de.std():.3f} eV")

    deg = np.minimum(rows["lumo_gap_A_eV"], rows["lumo_gap_B_eV"])
    print(f"  smallest LUMO/LUMO+1 gap: {deg.min():.3f} eV")
    if deg.min() < 0.1:
        print("    WARNING: near-degenerate frontier orbitals in some frames, "
              "where a single-orbital coupling understates transport - those "
              "frames need the 2x2 frontier block diagonalised instead")
    print(f"  mean wall time per frame: {rows['seconds'].mean() / 60:.1f} min")

if len(MOTIFS) > 1:
    print(f"\n{'motif':<14} {'rms |J| LUMO':>13} {'rms |J| HOMO':>13}")
    for motif, rows in summary.items():
        jl = np.sqrt((rows["J_lumo_meV"] ** 2).mean())
        jh = np.sqrt((rows["J_homo_meV"] ** 2).mean())
        print(f"{motif:<14} {jl:10.1f} meV {jh:10.1f} meV")
    print("\nThe larger contact area of H_backbone does not have to win: the "
          "LUMO of a\nnon-fullerene acceptor is concentrated on the terminal "
          "groups, which is\nexactly what the J_end_group contact brings "
          "together.")

## Save

`J_<motif>.txt` holds the LUMO-LUMO effective couplings in meV, one per frame.
The signed value is written rather than the magnitude: the sign is arbitrary,
since it follows the phase convention of the two monomer orbitals and that is
not fixed between separate SCFs, but keeping it throws nothing away and every
downstream use squares it anyway.

The header lines start with `#`, so `np.loadtxt` reads the column straight
back.

In [ ]:
# ---- J_<motif>.txt ------------------------------------------------------
for motif, rows in summary.items():
    j_lumo = np.atleast_1d(rows["J_lumo_meV"])
    path = f"J_{motif}.txt"
    with open(path, "w") as f:
        f.write(f"# LUMO-LUMO effective transfer integrals, meV\n")
        f.write(f"# AQx-2 {motif}, {XC}/{BASIS}, mode {MODE}, "
                f"{len(j_lumo)} frames\n")
        for j in j_lumo:
            f.write(f"{j:.4f}\n")
    print(f"{path}: {len(j_lumo)} values, rms "
          f"{np.sqrt((j_lumo ** 2).mean()):.1f} meV")